# 09d-ownership — Vì sao các lớp mới sở hữu cụm như vậy, và camera có chia cụm không

**Câu hỏi**: với nghiệm k-means của pipeline ở bước 09, lớp nào sở hữu cụm nào và vì sao (hiệu ứng đồng đều, mục tiêu SSE hay tối ưu); và trong từng lớp, cụm có bám camera hay cuộn không? (Q09.13; QD24, QD26)

**Vị trí**: phân tích (`lab/README.vi.md` §4 loại 3), dựa trên `09-cluster` (`8982942`) và cache landscape của nó; không đổi kết quả của bước 09. Đọc nhãn sau tiền đăng ký E09.07 (`1ce3233`); cell 0 dừng nếu thứ tự này sai.

**Quy tắc nhãn (QD26)**: chỉ dùng nhãn để mô tả nghiệm của pipeline và đo trần của mục tiêu k-means (nghiệm oracle). Không chấm bằng nhãn các cực tiểu khác của landscape; đó là việc của E09.07.

**Kế hoạch, chốt trước khi chạy** (2026-11-04):
1. Kiểm và đầu vào: lớp, camera, cuộn, vai trò của từng ảnh pool; `score09` trùng khoá `k5_*` của E09.05 v2.
2. Thành phần cụm: bảng cụm × lớp, sở hữu cụm, 50 ảnh cũ rơi vào đâu, ảnh bị xếp vào cụm của lớp khác có hộp lỗi của lớp chủ cụm không; hiệu ứng đồng đều (CV cỡ lớp so với CV cỡ cụm; Xiong, Wu & Chen 2009).
3. Sổ Ward: để một lớp nhỏ có cụm riêng, phải gộp hai cụm (tốn ΔJ = n_a·n_b/(n_a + n_b)·‖μ_a − μ_b‖²; Ward 1963) và tách lớp đó khỏi các cụm đang chứa nó (lợi ΔJ). Tính đúng, không chạy lại Lloyd, nên chỉ là cận.
4. Nghiệm oracle: phân hoạch theo 5 lớp mới, 50 ảnh cũ gán về tâm lớp mới gần nhất; inertia J_oracle. Chạy Lloyd từ tâm 5 lớp, được J_basin và ARI(basin, oracle) trên 698 ảnh lớp mới. So với J_best (inertia tốt nhất trong 10.000 lần khởi tạo đơn, `09-cluster` §3.1). Ba loại, chốt trước:
   - (a) J_basin < J_best: lỗi tối ưu. Có cực tiểu sâu hơn, gần lớp thật, mà k-means++ không tìm ra.
   - (b) J_basin ≥ J_best và ARI(basin, oracle) ≥ 0.8: lớp thật là một cực tiểu nhưng không sâu nhất. Mục tiêu SSE không ưa lớp thật; thêm khởi tạo không giúp.
   - (c) ARI(basin, oracle) < 0.8: lớp thật không phải điểm dừng của Lloyd. Lệch giữa mục tiêu và hình học dữ liệu.
   Kèm theo: nhóm nghiệm của basin trong landscape và khối lượng best-of-10 của nhóm đó.
5. Độ bất ổn từng ảnh d_i (best-of-10, `09-cluster` §3.3) theo lớp.
6. Q09.13 trong từng lớp: Cramér's V hiệu chỉnh chệch giữa cụm và camera, giữa cụm và cuộn; hoán vị phân tầng (camera trong cuộn, cuộn trong camera); bảng cuộn × camera; R² hiệu chỉnh của thành viên cụm, so với `05a`; lợi Ward khi chia silk_spot theo camera, theo cuộn, so với cách chia của k-means. Đối chứng âm: crease, rolled_pit, waist_folding.
7. Cụm crease: r0 s2 so với r0 s4 (QD24), so trên cùng một pool.

**Dự đoán chốt trước** [SUY LUẬN, xác suất chủ quan]:
- ở 15/15 ảnh chụp, chi phí gộp hai cụm lớn hơn lợi khi tách rolled_pit (khoảng 85%);
- với crease và inclusion, chi phí và lợi gần nhau, nên hai lớp này thay nhau giữ cụm thứ năm (khoảng 60%);
- oracle rơi vào loại (b) hoặc (c) ở ít nhất 2/3 FT seed (khoảng 65%);
- trong silk_spot, cụm bám cuộn ít nhất bằng bám camera (khoảng 55%);
- đối chứng âm không vượt mức nền (khoảng 80%).

### Nguồn gốc (provenance)

Mẫu cell 0 (`lab/README.vi.md` §11), cộng phép chặn của QD26: commit tiền đăng ký E09.07 `1ce3233` phải là tổ tiên của HEAD trước khi đọc nhãn.

In [1]:
# cell 0: provenance (run first); QD26 guard: labels are read only after the E09.07 pre-registration
import os, subprocess, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path[:0] = [str(ROOT), str(ROOT / "lab")]
import replay; os.environ["XLA_FLAGS"] = replay._xla_flags()   # GPU deterministic mode before any jax import
git = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.strip()
print("commit", git("rev-parse", "--short", "HEAD"),
      "| code dirty", bool(git("status", "--porcelain", "--", ".", ":(exclude)runs", ":(exclude)notes",
                               ":(exclude)lab/*.ipynb", ":(exclude)lab/*.md")),
      "| XLA_FLAGS", os.environ["XLA_FLAGS"])
import sklearn
pre_ok = subprocess.run(["git", "merge-base", "--is-ancestor", "1ce3233", "HEAD"]).returncode == 0
print("scikit-learn", sklearn.__version__, "| E09.07 pre-registration 1ce3233 is an ancestor of HEAD:", pre_ok)
assert pre_ok, "QD26: E09.07 must be pre-registered (1ce3233) before this notebook reads labels"

commit d282b37 | code dirty False | XLA_FLAGS --xla_gpu_deterministic_ops=true
scikit-learn 1.5.2 | E09.07 pre-registration 1ce3233 is an ancestor of HEAD: True


- **Kết quả**: commit `d282b37` (commit của kế hoạch chốt trước), code dirty False, XLA_FLAGS `--xla_gpu_deterministic_ops=true`; scikit-learn 1.5.2; commit tiền đăng ký E09.07 `1ce3233` có trong lịch sử của HEAD.

## 1. Kiểm và đầu vào

### 1.1 Lớp, camera, cuộn và vai trò của từng ảnh pool

Gắn mỗi hàng `pool_x` của 15 ảnh chụp với ảnh của nó theo byte trong file đặc trưng train (như `05a`, `09-cluster` §4.1), rồi đọc đủ các cột của `data_io.images`: lớp, vai trò, camera, cuộn. Lớp đọc từ file phải trùng `true_pool_y` của ảnh chụp. Mỗi ảnh chụp có một bảng `POOL[run]`: lớp, camera, cuộn, vai trò, id cụm của pipeline.

Dự đoán:
- lớp trùng `true_pool_y` ở 15/15;
- pool r0 s4 gồm silk 420, inclusion 135, waist 94, crease 32, rolled 17, và 50 ảnh cũ, mỗi lớp cơ sở 10 ảnh (E09.04);
- vai trò: S1 698, reserve 50.

In [2]:
# cell 1: inputs with labels (QD26: after the E09.07 pre-registration): every pool row of the 15 snapshots mapped to
#         its image (class, role, camera, coil) by its bytes in the train feature file; class checked against true_pool_y
import json, importlib
import numpy as np, pandas as pd
import data_io, landscape as ls
RUNS = [f"gc10_v2_ft_r{r}_s{s}" for r in range(3) for s in range(5)]
RUN, RUN_CREASE = "gc10_v2_ft_r0_s4", "gc10_v2_ft_r0_s2"            # QD24: illustration; crease mechanism
split_11 = json.load(open(ROOT / importlib.import_module("dataloaders.gc10").SPLIT))
NAMES = [c.split("_")[0] for c in split_11["classes"]]
img_11 = data_io.images("gc10")
tr_11 = img_11[img_11.part == "train"].set_index("row")
POOL, pos_11 = {}, {}
for run in RUNS:
    b = replay.snapshots(run)
    a = b["04"]["meta"]["args"]
    F = Path(a["data_dir"]) / a["pretrained_model_name"]
    if F not in pos_11:
        X = np.load(F / f"features-{F.name}.npy")
        pos_11[F] = {r.tobytes(): i for i, r in enumerate(X)}
    x = np.asarray(b["04"]["out"]["pool_x"])
    df = tr_11.loc[[pos_11[F][r.tobytes()] for r in x], ["path", "cls", "role", "cam", "coil"]].reset_index()
    df["y"] = np.asarray(b["04"]["out"]["true_pool_y"]).astype(int)
    df["cluster"] = np.asarray(b["09"]["out"]["ids"]).astype(int)
    POOL[run] = df
print("class from file == true_pool_y:", sum(bool((d.cls == d.y).all()) for d in POOL.values()), "/ 15 | rows per snapshot:",
      sorted({len(d) for d in POOL.values()}))
print("names:", NAMES)
d_11 = POOL[RUN]
print("r0 s4 pool per class:", d_11.y.map(lambda c: NAMES[c]).value_counts().to_dict())
print("roles:", d_11.role.value_counts().to_dict(), "| old images per base class:",
      d_11[d_11.y < 5].y.map(lambda c: NAMES[c]).value_counts().to_dict())

/home/rito/miniforge3/envs/vbcgcd/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


class from file == true_pool_y: 15 / 15 | rows per snapshot: [748]
names: ['punching', 'welding', 'crescent', 'water', 'oil', 'silk', 'inclusion', 'rolled', 'crease', 'waist']
r0 s4 pool per class: {'silk': 420, 'inclusion': 135, 'waist': 94, 'crease': 32, 'rolled': 17, 'punching': 10, 'crescent': 10, 'welding': 10, 'oil': 10, 'water': 10}
roles: {'S1': 698, 'reserve': 50} | old images per base class: {'punching': 10, 'crescent': 10, 'welding': 10, 'oil': 10, 'water': 10}


- **Kết quả**:
  - Lớp đọc từ file trùng `true_pool_y` ở 15/15 ảnh chụp, mỗi ảnh chụp 748 hàng.
  - Pool r0 s4: silk 420, inclusion 135, waist 94, crease 32, rolled 17; 50 ảnh cũ, đúng 10 ảnh cho mỗi lớp cơ sở (punching, welding, crescent, water, oil).
  - Vai trò: S1 698, reserve 50.
- **Ý nghĩa**: đúng như dự đoán; đầu vào có nhãn khớp ảnh chụp và E09.04.

### 1.2 Bảng tầng 1 của 15 nghiệm pipeline

Tính lại `score09` của `lab/score.py` (sở hữu theo lớp đông nhất, kể cả lớp cũ; best(c); balanced accuracy ghép một–một; AMI; độ thuần) trên nghiệm pipeline của 15 ảnh chụp. So với các khoá `k5_*` mà E09.05 v2 đã ghi trong `metrics.json` của `gc10_e09_05_v2_r<r>_s<N>`; kèm `k5_inertia` so với inertia của ảnh chụp.

Dự đoán:
- mọi khoá trùng ở 15/15;
- sở hữu (silk, inclusion, rolled, crease, waist) như QD24: waist luôn 1 cụm, rolled không cụm nào, silk 2–3 cụm; cụm còn lại về crease hoặc inclusion.

In [3]:
# cell 2: tier-1 scores of the 15 pipeline partitions (lab/score.py score09) against the k5_* keys written by E09.05 v2
import score
rows_12 = []
for run in RUNS:
    d = POOL[run]
    sc = score.score09(d.y.values, d.cluster.values, NAMES, n_old=5)
    m = json.load(open(next(Path("runs").glob(f"*__gc10_e09_05_v2_{run[11:]}")) / "metrics.json"))
    keys = [k for k in m if k.startswith("k5_") and k != "k5_inertia"]
    bad = [k for k in keys if not np.isclose(m[k], sc[k[3:]], rtol=0, atol=1e-12)]
    rows_12.append(dict(run=run[11:], keys=len(keys), mismatched=len(bad),
                        inertia_ok=bool(np.isclose(m["k5_inertia"], ls.pool(run)["inertia"], rtol=1e-6)),
                        bal_acc=round(sc["bal_acc_hungarian"], 3), owned_s_i_r_c_w=[sc[f"owned_{n}"] for n in NAMES[5:]],
                        best_c=round(sc["best_crease"], 2), best_i=round(sc["best_inclusion"], 2)))
print(pd.DataFrame(rows_12).to_string(index=False))

  run  keys  mismatched  inertia_ok  bal_acc owned_s_i_r_c_w  best_c  best_i
r0_s0    26           0        True    0.575 [2, 1, 0, 1, 1]    0.88    0.32
r0_s1    26           0        True    0.569 [2, 1, 0, 1, 1]    0.84    0.32
r0_s2    26           0        True    0.571 [3, 0, 0, 1, 1]    0.88    0.00
r0_s3    26           0        True    0.315 [3, 1, 0, 0, 1]    0.00    0.34
r0_s4    26           0        True    0.317 [3, 1, 0, 0, 1]    0.00    0.34
r1_s0    26           0        True    0.529 [3, 0, 0, 1, 1]    0.66    0.00
r1_s1    26           0        True    0.524 [3, 0, 0, 1, 1]    0.66    0.00
r1_s2    26           0        True    0.532 [3, 0, 0, 1, 1]    0.66    0.00
r1_s3    26           0        True    0.351 [3, 1, 0, 0, 1]    0.00    0.31
r1_s4    26           0        True    0.406 [3, 1, 0, 0, 1]    0.00    0.71
r2_s0    26           0        True    0.458 [2, 2, 0, 0, 1]    0.00    0.61
r2_s1    26           0        True    0.350 [3, 1, 0, 0, 1]    0.00    0.30

- **Kết quả**:
  - Cả 26 khoá `k5_*` trùng E09.05 v2 ở 15/15 ảnh chụp (0 khoá lệch); `k5_inertia` trùng inertia của ảnh chụp.
  - Sở hữu cụm (silk, inclusion, rolled, crease, waist) đúng như QD24: waist luôn 1 cụm, rolled không cụm nào, silk 2 hoặc 3 cụm. Cụm còn lại về crease ở 4/12 ảnh chụp mà silk có 3 cụm, về inclusion ở 8/12.
  - Balanced accuracy trên pool: 0.52–0.58 khi crease có cụm (r0 s0–s2, r1 s0–s2); 0.31–0.46 khi crease không có cụm.
- **Ý nghĩa**: đúng như dự đoán. Bảng tầng 1 của 15 nghiệm pipeline tái lập được từ ảnh chụp, nên các mục sau dùng đúng những nghiệm mà E09.05 đã chấm. Các số này đã thấy trước tiền đăng ký E09.07 (ghi ở đó).

## 2. Thành phần cụm

### 2.1 Bảng cụm × lớp và nơi mỗi lớp rơi vào

- r0 s4: in đủ bảng cụm × 10 lớp, kèm lớp chủ cụm (lớp đông nhất, tính cả lớp cũ, như `score09`), cỡ cụm và độ thuần.
- Qua 15 ảnh chụp: với mỗi lớp mới, phần lớn ảnh của nó nằm trong cụm do lớp nào sở hữu, và bao nhiêu ảnh; 50 ảnh cũ nằm trong cụm của những lớp nào.

Dự đoán [SUY LUẬN]:
- ở r0 s4, silk sở hữu 3 cụm; cụm của waist chứa phần lớn crease và rolled (theo E09.05: 27/32 crease, 16/17 rolled);
- ở các ảnh chụp không có cụm inclusion, inclusion rơi chủ yếu vào cụm của silk;
- 50 ảnh cũ rơi chủ yếu vào các cụm silk, vì đó là các cụm lớn nhất.

In [4]:
# cell 3: composition of the pipeline's clusters: cluster x class table (r0 s4 in full), ownership (plurality class, old
#         classes included, as score09), where most images of each novel class and the 50 old images go, 15 snapshots
def table_21(d):
    t = pd.crosstab(d.cluster, d.y.map(lambda c: NAMES[c])).reindex(columns=NAMES, fill_value=0)
    return t, t.idxmax(axis=1)                                        # table (clusters x classes), owner class per cluster

t_21, own_21 = table_21(POOL[RUN])
show_21 = t_21.assign(owner=own_21, size=t_21.sum(axis=1), purity=(t_21.max(axis=1) / t_21.sum(axis=1)).round(2))
print("r0 s4, rows = clusters of the pipeline:")
print(show_21.to_string())
rows_21 = []
for run in RUNS:
    d = POOL[run]
    _, own = table_21(d)
    host = d.cluster.map(own)                                         # owner class of each image's cluster
    row = dict(run=run[11:])
    for n in NAMES[5:]:
        vc = host[d.y.map(lambda c: NAMES[c]) == n].value_counts()
        row[n] = f"{vc.index[0]} {vc.iloc[0]}/{vc.sum()}"
    row["old images"] = host[d.y < 5].value_counts().to_dict()
    rows_21.append(row)
print("\nfor each novel class: owner of the cluster holding most of its images")
print(pd.DataFrame(rows_21).to_string(index=False))

r0 s4, rows = clusters of the pipeline:
y        punching  welding  crescent  water  oil  silk  inclusion  rolled  crease  waist      owner  size  purity
cluster                                                                                                          
0               0        1         6      3    6    29         46       1       3      0  inclusion    95    0.48
1              10        9         4      0    0     1         11      16      27     79      waist   157    0.50
2               0        0         0      6    4   169         76       0       2     15       silk   272    0.62
3               0        0         0      1    0   118          2       0       0      0       silk   121    0.98
4               0        0         0      0    0   103          0       0       0      0       silk   103    1.00

for each novel class: owner of the cluster holding most of its images
  run         silk         inclusion       rolled       crease       waist                 

- **Kết quả**:
  - r0 s4 có 5 cụm:
    - silk sở hữu 3 cụm: một cụm lẫn (169 silk + 76 inclusion, độ thuần 0.62) và hai cụm gần như toàn silk (118/121 và 103/103);
    - cụm của waist (157 ảnh, độ thuần 0.50) chứa 79 waist, 27/32 crease, 16/17 rolled và 19 ảnh cũ (10 punching, 9 welding);
    - cụm của inclusion (95 ảnh, độ thuần 0.48) chứa 46 inclusion, 29 silk và 16 ảnh cũ.
  - Qua 15 ảnh chụp:
    - silk: 323–420/420 ảnh nằm trong cụm silk;
    - waist: 78–94/94 ảnh nằm trong cụm waist;
    - inclusion: khi không có cụm riêng, 119–126/135 ảnh nằm trong cụm silk; khi có cụm riêng, vẫn 78–86/135 ảnh nằm trong cụm silk ở 5 ảnh chụp, và chỉ 67–123/135 ảnh nằm trong cụm của chính nó;
    - crease và rolled đi cùng nhau: ở 6 ảnh chụp có cụm crease, cụm đó chứa 21–28/32 crease và 13–15/17 rolled; ở 9 ảnh chụp còn lại, cả hai nằm trong cụm của waist (crease 21–28/32, rolled 14–16/17).
  - 50 ảnh cũ: 20–36 ảnh nằm trong cụm của crease hoặc waist, phần còn lại trong cụm silk hoặc inclusion.
- **Ý nghĩa**:
  - Dự đoán đúng ở r0 s4 và ở inclusion. Sai ở ảnh cũ: chúng rơi chủ yếu vào cụm crease/waist chứ không phải cụm silk.
  - Cụm thứ năm (crease hay inclusion) thực ra là "crease cùng rolled" hoặc "inclusion lẫn silk". rolled_pit không bao giờ đứng một mình; nó luôn đi theo crease.

### 2.2 Nhãn mơ hồ trong pool và hiệu ứng đồng đều

(a) Ảnh lớp mới nằm trong cụm do một lớp khác sở hữu ("xếp sai"). Những ảnh này có hộp lỗi của lớp chủ cụm trong XML không (`data_io.objects`), và có nhiều loại lỗi không? Bài toán giống 08 §4.1, nhưng làm trên pool. Bảng theo ảnh chụp, rồi cộng 15 ảnh chụp theo cặp (lớp thật, lớp chủ cụm).

(b) Hiệu ứng đồng đều (Xiong, Wu & Chen 2009): CV của cỡ 5 lớp mới trong pool so với CV của cỡ 5 cụm.

Dự đoán [SUY LUẬN]:
- ảnh xếp sai nhiều loại lỗi hơn ảnh xếp đúng;
- tỷ lệ có hộp của lớp chủ cụm thấp ở cặp inclusion → silk, vì inclusion hay đi với water_spot, một lớp cũ (`00-data-gc10`);
- CV cỡ lớp 1.17, CV cỡ cụm 0.24–0.57 (`09-cluster` §4.2).

In [5]:
# cell 4: (a) label ambiguity in the pool: novel images placed in a cluster owned by another class -- multi-defect? a box
#         of the owner class in their XML (data_io.objects)? (b) uniform effect: CV of class sizes vs CV of cluster sizes
OBJ_22 = {}
def boxes_22(path):                                                   # (classes with a box, more than one defect type)
    if path not in OBJ_22:
        ob = data_io.objects("gc10", path) or []
        OBJ_22[path] = ({o["cls"] for o in ob if o["cls"] is not None}, len({o["name"] for o in ob}) > 1)
    return OBJ_22[path]

rows_22, pairs_22 = [], {}
for run in RUNS:
    d = POOL[run]
    _, own = table_21(d)
    nov = d[d.y >= 5].copy()
    nov["host"] = nov.cluster.map(own).map(NAMES.index)
    nov["mis"] = nov.host != nov.y
    bx = nov.path.map(boxes_22)
    nov["multi"] = bx.map(lambda b: b[1])
    nov["host_box"] = [h in b[0] for h, b in zip(nov.host, bx)]
    m = nov[nov.mis]
    rows_22.append(dict(run=run[11:], novel=len(nov), misplaced=len(m), multi_misplaced=int(m.multi.sum()),
                        host_box_misplaced=int(m.host_box.sum()),
                        multi_placed=f"{int(nov[~nov.mis].multi.sum())}/{int((~nov.mis).sum())}"))
    for (yy, hh), g in m.groupby(["y", "host"]):
        k = (NAMES[yy], NAMES[hh])
        a0, a1, a2 = pairs_22.get(k, (0, 0, 0))
        pairs_22[k] = (a0 + len(g), a1 + int(g.multi.sum()), a2 + int(g.host_box.sum()))
print(pd.DataFrame(rows_22).to_string(index=False))
print("\nsummed over 15 snapshots, by (true class, owner of its cluster):")
print(pd.DataFrame([dict(true=k[0], host=k[1], n=v[0], multi=v[1], host_box=v[2])
                    for k, v in sorted(pairs_22.items(), key=lambda kv: -kv[1][0])]).to_string(index=False))
n_c22 = POOL[RUN][POOL[RUN].y >= 5].y.value_counts().sort_index().to_numpy()
cv1_22 = [np.bincount(POOL[r].cluster).std(ddof=1) / np.bincount(POOL[r].cluster).mean() for r in RUNS]
print(f"\nuniform effect: CV of the 5 novel class sizes {n_c22.std(ddof=1) / n_c22.mean():.2f} (sizes {n_c22.tolist()}) | "
      f"CV of the 5 cluster sizes {min(cv1_22):.2f}-{max(cv1_22):.2f}")

  run  novel  misplaced  multi_misplaced  host_box_misplaced multi_placed
r0_s0    698        156               15                   7       10/542
r0_s1    698        157               15                   7       10/541
r0_s2    698        168               20                  12        5/530
r0_s3    698        183               15                   5       10/515
r0_s4    698        183               15                   4       10/515
r1_s0    698        177               21                  11        4/521
r1_s1    698        177               21                  11        4/521
r1_s2    698        178               21                  11        4/520
r1_s3    698        158               16                   6        9/540
r1_s4    698        186               12                   7       13/512
r2_s0    698        152                5                   3       20/546
r2_s1    698        158               16                   6        9/540
r2_s2    698        132               

- **Kết quả**:
  - Có 132–186/698 ảnh lớp mới bị xếp vào cụm của lớp khác.
  - Trong các ảnh xếp sai, 5–21 ảnh có nhiều loại lỗi (3–13%); trong các ảnh xếp đúng chỉ 4–20/512–566 ảnh (khoảng 1–4%).
  - Ảnh xếp sai có hộp lỗi của lớp chủ cụm: chỉ 3–12 ảnh mỗi ảnh chụp, tổng 110/2471 (4.5%).
  - Cộng 15 ảnh chụp theo cặp (lớp thật → lớp chủ cụm):
    - inclusion → silk: 1183 lượt, trong đó 168 nhiều loại lỗi và 92 có hộp silk (7.8%);
    - silk → inclusion: 424, 0 có hộp inclusion;
    - crease → waist: 238, 0 có hộp waist;
    - rolled → waist: 143, 0;
    - rolled → crease: 87, 0.
  - Hiệu ứng đồng đều: CV cỡ 5 lớp mới là 1.17 (420, 135, 17, 32, 94), CV cỡ 5 cụm là 0.24–0.57.
- **Ý nghĩa**:
  - Khác 08 §4.1: ở test S0, 17/35 lỗi có hộp của lớp bị đoán thành. Còn ở pool, xếp sai gần như không phải do nhãn mơ hồ (4.5%). Đúng là ảnh xếp sai nhiều loại lỗi hơn, nhưng chúng chỉ chiếm phần nhỏ.
  - [SUY LUẬN] Xếp sai ở bước 09 là chuyện cấu trúc của mục tiêu k-means: crease, rolled bị gộp vào cụm waist; inclusion bị gộp vào cụm silk. Đúng dự đoán về cặp inclusion → silk (tỷ lệ có hộp silk thấp) và về hiệu ứng đồng đều.

## 3. Sổ Ward

### 3.1 Chi phí giải phóng một cụm so với lợi khi tách một lớp

Trên nghiệm của pipeline, ở không gian 768 chiều thô của bước 09:
- **chi phí giải phóng một cụm** = chi phí gộp rẻ nhất trong 10 cặp cụm, ΔJ = n_a·n_b/(n_a + n_b)·‖μ_a − μ_b‖² (Ward 1963); kèm tên lớp chủ của hai cụm bị gộp;
- **lợi khi tách lớp c** = J hiện tại − J sau khi chuyển mọi ảnh của c ra một cụm riêng, các cụm khác giữ nguyên thành viên (tính đúng SSE); cho rolled, crease, crease + rolled, inclusion;
- **net** = lợi − chi phí.

**Cách làm**: sau phép hoán đổi, chạy Lloyd chỉ có thể giảm J thêm, nên J của phương án thay thế ≤ J − net. Vì vậy:
- net > 0: phương án thay thế chắc chắn có SSE thấp hơn;
- net < 0: chưa đủ để kết luận; mục 4 chạy Lloyd thật.

Dự đoán [SUY LUẬN, đã chốt ở cell đầu]:
- net của rolled âm ở 15/15 (17 ảnh thì lợi nhỏ);
- net của crease và của inclusion gần 0, vì hai lớp này thay nhau giữ cụm thứ năm;
- cặp bị gộp rẻ nhất là hai cụm silk ở các ảnh chụp mà silk có 3 cụm.

In [6]:
# cell 5: Ward ledger on the pipeline partition (raw pool features): cost of freeing one cluster (cheapest merge of two
#         clusters, Ward 1963) vs gain of moving one class into a cluster of its own; exact SSE, no Lloyd step (a bound)
def sse_31(X):
    return float(((X - X.mean(0)) ** 2).sum()) if len(X) else 0.0

rows_31 = []
for run in RUNS:
    d = POOL[run]
    x, ids = ls.pool(run)["x"].astype(np.float64), d.cluster.values
    K, n = ids.max() + 1, np.bincount(ids)
    mu = np.stack([x[ids == k].mean(0) for k in range(K)])
    J = sum(sse_31(x[ids == k]) for k in range(K))
    _, own = table_21(d)
    cost, a, b = min((n[a] * n[b] / (n[a] + n[b]) * ((mu[a] - mu[b]) ** 2).sum(), a, b) for a in range(K) for b in range(a + 1, K))
    row = dict(run=run[11:], J=round(J), merge_cost=round(cost), merged=f"{own[a]}+{own[b]}")
    for name, cls in (("rolled", [7]), ("crease", [8]), ("crease+rolled", [7, 8]), ("inclusion", [6])):
        m = d.y.isin(cls).values
        row[f"gain_{name}"] = round(J - sum(sse_31(x[(ids == k) & ~m]) for k in range(K)) - sse_31(x[m]))
    rows_31.append(row)
df_31 = pd.DataFrame(rows_31)
for name in ("rolled", "crease", "crease+rolled", "inclusion"):
    df_31[f"net_{name}"] = df_31[f"gain_{name}"] - df_31["merge_cost"]
print(df_31.to_string(index=False))

  run     J  merge_cost         merged  gain_rolled  gain_crease  gain_crease+rolled  gain_inclusion  net_rolled  net_crease  net_crease+rolled  net_inclusion
r0_s0 82352        4231   crease+waist          659         1160                 811           -1293       -3572       -3071              -3420          -5524
r0_s1 82649        4155   crease+waist          566         1274                 835           -1245       -3589       -2881              -3320          -5400
r0_s2 83113        3645      silk+silk          709         1209                 886              32       -2936       -2436              -2759          -3613
r0_s3 82478        3714      silk+silk          765         1709                1584           -1124       -2949       -2005              -2130          -4838
r0_s4 82926        3746      silk+silk          766         1692                1572            -946       -2980       -2054              -2174          -4692
r1_s0 89442        4625      silk+silk        

- **Kết quả**:
  - Chi phí giải phóng một cụm: 3645–4748. Cặp bị gộp rẻ nhất là hai cụm silk ở 10/15 ảnh chụp; crease + waist ở r0 s0, s1 (nơi crease có cụm và silk có 2 cụm); một cụm silk với cụm inclusion ở 3 ảnh chụp.
  - Lợi khi tách một lớp ra cụm riêng: rolled 566–998; crease 1160–1709; crease + rolled 811–1676; inclusion từ −1293 đến +1486. Lợi âm nghĩa là gom hết inclusion vào một cụm còn làm SSE tăng.
  - Net (lợi − chi phí) âm ở cả 60 ô: rolled từ −2936 đến −3856; crease từ −2005 đến −3392; crease + rolled từ −2130 đến −3420; inclusion từ −3084 đến −5524.
- **Ý nghĩa**:
  - Chi phí giải phóng một cụm lớn gấp 2–4 lần lợi tách một lớp nhỏ. Không phép hoán đổi một bước nào làm SSE giảm.
  - Dự đoán đúng với rolled (âm ở 15/15). Sai với crease và inclusion: net không gần 0 mà âm rõ.
  - Đây chỉ là cận, vì chưa chạy Lloyd sau khi hoán đổi; câu trả lời có Lloyd ở mục 4.
  - [SUY LUẬN] Với K = 5 và cỡ lớp lệch như GC10, SSE "thích" dành cụm cho phần lớn và tản mát của silk hơn là cho một lớp nhỏ, đúng cơ chế của hiệu ứng đồng đều.

## 4. Nghiệm oracle

### 4.1 Lớp thật có phải cực tiểu của k-means không (ba loại a/b/c đã chốt)

- **Phân hoạch oracle**: 5 lớp mới, 50 ảnh cũ gán về tâm lớp mới gần nhất; J_oracle tính đúng.
- **Basin**: chạy Lloyd (`KMeans(init=5 tâm lớp, n_init=1)`) trên cả pool, được J_basin.
- Đo:
  - ARI(basin, oracle) và ARI(basin, nghiệm pipeline), trên 698 ảnh lớp mới;
  - mỗi lớp còn bao nhiêu ảnh ở lại cụm xuất phát từ tâm của nó;
  - nhóm nghiệm (ngưỡng 0.95) của basin trong landscape, kèm khối lượng best-of-10 của nhóm đó;
  - so J_basin với J_best (inertia tốt nhất trong 10.000 lần khởi tạo đơn) và với J_pipe.
- Ba loại kết cục, đã chốt ở cell đầu:
  - (a) J_basin < J_best;
  - (b) J_basin ≥ J_best và ARI(basin, oracle) ≥ 0.8;
  - (c) ARI(basin, oracle) < 0.8.

Dự đoán [SUY LUẬN]:
- J_oracle > J_pipe ở mọi ảnh chụp;
- trong basin, rolled tan gần hết, crease giữ một phần, silk và waist giữ gần hết;
- loại (b) hoặc (c) ở ít nhất 2/3 FT seed, gần như không có (a).

In [7]:
# cell 6: oracle partition (5 novel classes; old images to the nearest novel class mean) and Lloyd from the 5 class means on
#         the full pool; the basin located in the landscape (modes at ARI 0.95); categories a/b/c fixed in the title cell
import multiprocessing as mp, time
from sklearn.cluster import KMeans
t0_41 = time.time()
MD95 = ls.modes_all(RUNS, n=10000, taus=(0.95,), workers=max(1, (os.cpu_count() or 2) - 2))
print(f"modes at 0.95: {(time.time() - t0_41) / 60:.1f} min")
rows_41, BASIN = [], {}
for run in RUNS:
    d = POOL[run]
    p = ls.pool(run)
    x, y = p["x"], d.y.values
    nov = y >= 5
    mu = np.stack([x[y == c].mean(0) for c in range(5, 10)])
    lab_or = np.where(nov, y - 5, ((x[:, None, :].astype(np.float64) - mu[None]) ** 2).sum(-1).argmin(1))
    J_or = sum(sse_31(x[lab_or == k].astype(np.float64)) for k in range(5))
    km = KMeans(n_clusters=5, init=mu, n_init=1).fit(x)
    lab_b = km.labels_
    BASIN[run] = lab_b
    s, _ = ls.singles(run, n=10000)
    md = MD95[(run, 0.95)]
    bm, ba = ls.assign(lab_b, s["L"], md, 0.95)
    w10 = ls.best_of_weights(s["J"], 10)
    ari_bo = ls.ari(lab_b[nov], y[nov] - 5)
    J_b, J_best = float(km.inertia_), float(s["J"].min())
    cat = "a" if J_b < J_best * (1 - 1e-6) else ("b" if ari_bo >= 0.8 else "c")
    row = dict(run=run[11:], J_oracle=round(J_or), J_basin=round(J_b), J_best=round(J_best), J_pipe=round(p["inertia"]),
               ari_basin_oracle=round(ari_bo, 3), ari_basin_pipe=round(ls.ari(lab_b[nov], d.cluster.values[nov]), 3),
               basin_mode=bm, basin_max_ari=round(ba, 3), P10_basin_mode=round(float(w10[md["mode"] == bm].sum()) if bm >= 0 else 0.0, 3),
               n_iter=km.n_iter_, category=cat)
    for c in range(5, 10):
        row[f"keep_{NAMES[c]}"] = f"{int((lab_b[y == c] == c - 5).sum())}/{int((y == c).sum())}"
    rows_41.append(row)
df_41 = pd.DataFrame(rows_41)
print(df_41.to_string(index=False))
print("\ncategories per FT seed:", {f"r{r}": df_41[df_41.run.str.startswith(f"r{r}")].category.value_counts().to_dict() for r in range(3)})

modes at 0.95: 0.5 min
  run  J_oracle  J_basin  J_best  J_pipe  ari_basin_oracle  ari_basin_pipe  basin_mode  basin_max_ari  P10_basin_mode  n_iter category keep_silk keep_inclusion keep_rolled keep_crease keep_waist
r0_s0     96398    83922   82271   82352             0.342           0.797         730          0.996             0.0       6        c   233/420         92/135       12/17       28/32      81/94
r0_s1     96827    84358   82609   82649             0.344           0.799         699          0.967             0.0       8        c   233/420         92/135       13/17       28/32      81/94
r0_s2     96998    84420   82684   83113             0.344           0.718         723          0.979             0.0       6        c   233/420         92/135       13/17       28/32      81/94
r0_s3     96492    83914   82288   82478             0.344           0.565         783          0.966             0.0       5        c   233/420         92/135       12/17       30/32      82/94
r0

- **Kết quả**:
  - J_oracle 96.398–107.221, cao hơn J_pipe khoảng 17%.
  - Lloyd chạy từ 5 tâm lớp hội tụ sau 5–27 vòng về J_basin 83.914–95.931, vẫn cao hơn J_best 0.9–3.6%. Không ảnh chụp nào thuộc loại (a).
  - ARI(basin, oracle) trên ảnh lớp mới chỉ 0.34–0.42, nên cả 15/15 ảnh chụp thuộc loại (c), ở cả 3 FT seed. ARI(basin, nghiệm pipeline) 0.51–0.81.
  - Trong basin, số ảnh mỗi lớp còn ở lại cụm xuất phát từ tâm của nó:
    - silk chỉ 226–243/420; waist 81–93/94;
    - inclusion 55–92/135;
    - crease: r0 28–30/32; r1 3–5/32 (r1 s2: 19); r2 16–24/32;
    - rolled: r0, r1 12–15/17; r2 s0, s2, s4 chỉ 5/17.
  - Basin có mặt trong landscape (ARI lớn nhất với một nghiệm đầu nhóm 0.95–1.00; riêng r2 s3 không thuộc nhóm nào), nhưng khối lượng best-of-10 của nhóm đó là 0.000 ở cả 15 ảnh chụp.
- **Ý nghĩa**:
  - Theo quy tắc đã chốt: loại (c). Phân hoạch theo lớp thật không phải điểm dừng của Lloyd, và mục tiêu SSE ở 768 chiều thô xa lớp thật hơn nhiều so với nghiệm pipeline.
  - ARI thấp chủ yếu do silk: gần một nửa ảnh silk rời cụm xuất phát (silk chiếm 420/698 ảnh nên chi phối ARI). Các lớp nhỏ thì không tan hết: ở r0, basin vẫn giữ một cụm crease (28–30/32) và phần lớn rolled.
  - Vậy ở r0 tồn tại một cực tiểu mà crease và rolled có cụm riêng, nhưng SSE của nó cao hơn nghiệm tốt nhất khoảng 2%, và best-of-10 hầu như không bao giờ rơi vào đó.
  - So với dự đoán: đúng ở J_oracle > J_pipe và ở loại (b)/(c) với không có (a). Sai ở chi tiết: rolled không tan ở r0, r1; silk không giữ được cụm của mình.
  - [SUY LUẬN] Thêm khởi tạo đẩy nghiệm về phía J_best, mà J_best ở xa phân hoạch theo lớp. Điều này nghiêng về giả thuyết ngược của E09.07, nhưng chỉ E09.07 ở tầng 2 mới quyết; thiết kế của nó đã chốt trước khi có số này (QD26).

## 5. Độ bất ổn theo lớp

### 5.1 Ảnh của lớp nào đổi cụm giữa các lần chạy pipeline

d_i được tính như ở `09-cluster` §3.3 (best-of-10 trên 10.000 lần khởi tạo đơn); ở đây lấy trung bình theo lớp, kể cả nhóm 50 ảnh cũ. Trung bình d_i của một lớp là xác suất trung bình để hai lần chạy pipeline độc lập bất đồng về việc một ảnh của lớp đó có chung cụm với một ảnh bất kỳ khác hay không.

Dự đoán [SUY LUẬN]:
- crease, rolled và inclusion bất ổn nhất, vì chúng là các lớp thay nhau giữ cụm thứ năm;
- waist ổn định nhất;
- silk ở giữa, vì cách chia silk thành 2–3 cụm đổi theo lần chạy.

In [8]:
# cell 7: per-image instability d_i of the pipeline's procedure (best-of-10 over the 10 000 single inits, 09-cluster §3.3)
#         averaged by class: which classes' images change cluster between independent runs
rows_51 = []
for run in RUNS:
    s, _ = ls.singles(run, n=10000)
    dI = ls.instability(s["L"], ls.best_of_weights(s["J"], 10))
    y = POOL[run].y.values
    row = dict(run=run[11:], all=dI.mean())
    row.update({NAMES[c]: dI[y == c].mean() for c in range(5, 10)})
    row["old"] = dI[y < 5].mean()
    rows_51.append(row)
df_51 = pd.DataFrame(rows_51)
print(df_51.round(3).to_string(index=False))
print("\nmean by FT seed:")
print(df_51.groupby(df_51.run.str[:2]).mean(numeric_only=True).round(3).to_string())

  run   all  silk  inclusion  rolled  crease  waist   old
r0_s0 0.085 0.093      0.096   0.051   0.056  0.049 0.080
r0_s1 0.077 0.077      0.071   0.058   0.066  0.088 0.084
r0_s2 0.090 0.097      0.104   0.059   0.067  0.056 0.089
r0_s3 0.082 0.082      0.081   0.062   0.078  0.088 0.079
r0_s4 0.085 0.085      0.087   0.068   0.088  0.085 0.082
r1_s0 0.041 0.033      0.064   0.074   0.064  0.021 0.056
r1_s1 0.050 0.032      0.074   0.098   0.093  0.038 0.105
r1_s2 0.039 0.032      0.061   0.067   0.061  0.017 0.059
r1_s3 0.050 0.032      0.080   0.108   0.098  0.044 0.089
r1_s4 0.037 0.030      0.059   0.069   0.062  0.020 0.044
r2_s0 0.088 0.087      0.146   0.039   0.061  0.015 0.105
r2_s1 0.063 0.064      0.098   0.036   0.055  0.012 0.070
r2_s2 0.083 0.078      0.143   0.055   0.070  0.020 0.103
r2_s3 0.072 0.075      0.110   0.039   0.054  0.013 0.079
r2_s4 0.064 0.062      0.116   0.034   0.051  0.011 0.066

mean by FT seed:
       all   silk  inclusion  rolled  crease  waist   

- **Kết quả**: trung bình d_i theo FT seed:

  | FT seed | chung | silk | inclusion | rolled | crease | waist | ảnh cũ |
  |---|---|---|---|---|---|---|---|
  | r0 | 0.084 | 0.087 | 0.088 | 0.060 | 0.071 | 0.073 | 0.083 |
  | r1 | 0.043 | 0.032 | 0.067 | 0.083 | 0.075 | 0.028 | 0.070 |
  | r2 | 0.074 | 0.073 | 0.123 | 0.041 | 0.058 | 0.014 | 0.085 |

- **Ý nghĩa**:
  - Hình dạng phụ thuộc FT seed:
    - r1: silk và waist ổn định, còn rolled, crease, inclusion bất ổn gấp 2–3 lần, đúng dự đoán;
    - r2: inclusion bất ổn nhất, waist gần như cố định; rolled và crease thấp hơn dự đoán;
    - r0: các lớp gần như bất ổn như nhau, kể cả waist và silk, sai dự đoán.
  - [SUY LUẬN] Ở r1 và r2, nhiễu của bước 09 dồn vào các lớp hiếm (r1) hoặc vào inclusion (r2). Balanced novel cân mỗi lớp như nhau (QD25), nên nhiễu này đi thẳng vào đại lượng chính. Đây là lý do E09.07 cần lấy trung bình nhiều lần rút.

## 6. Q09.13 trong từng lớp

### 6.1 Cụm có bám camera hay cuộn trong cùng một lớp không

Với mỗi lớp mới và mỗi ảnh chụp, chỉ xét ảnh của lớp đó:
- Cramér's V hiệu chỉnh chệch (Bergsma 2013) giữa id cụm của pipeline và camera, giữa id cụm và cuộn;
- ba loại p-value, mỗi loại 1000 lần hoán vị:
  - hoán vị thường;
  - camera hoán vị trong từng cuộn (camera có thêm gì ngoài cuộn không);
  - cuộn hoán vị trong từng camera (cuộn có thêm gì ngoài camera không);
- số cụm chứa ít nhất 3 ảnh của lớp; lớp nằm gọn trong một cụm thì V không xác định.

**Cách làm**: p = (1 + số lần hoán vị cho V ≥ V quan sát) / (1 + 1000). Nếu mỗi cuộn chỉ được chụp bởi một camera, hoán vị camera trong cuộn không đổi gì và p = 1. Điều đó đúng về logic: khi ấy dữ liệu không tách được camera khỏi cuộn.

Dự đoán [SUY LUẬN, đã chốt ở cell đầu]:
- với silk và inclusion, V vượt mức nền ở phép hoán vị thường;
- trong silk, cuộn | camera có ý nghĩa thống kê ít nhất thường xuyên bằng camera | cuộn;
- đối chứng âm (crease, rolled, waist): không vượt mức nền, hoặc V không xác định vì lớp nằm gọn trong một cụm.

In [9]:
# cell 8: Q09.13 within each novel class: association of the pipeline's cluster with camera and with coil (bias-corrected
#         Cramer's V), permutation p-values (1000): plain, camera permuted within coil, coil permuted within camera
def codes_61(a):
    return np.unique(np.asarray(a), return_inverse=True)[1]

def vcorr_61(a, b):                                   # bias-corrected Cramer's V of two integer code arrays
    r, k = a.max() + 1, b.max() + 1
    t = np.bincount(a * k + b, minlength=r * k).reshape(r, k).astype(float)
    t = t[t.sum(axis=1) > 0][:, t.sum(axis=0) > 0]
    n, (r, k) = t.sum(), t.shape
    if r < 2 or k < 2:
        return np.nan
    e = t.sum(axis=1, keepdims=True) * t.sum(axis=0, keepdims=True) / n
    phi2c = max(0.0, ((t - e) ** 2 / e).sum() / n - (k - 1) * (r - 1) / (n - 1))
    rc, kc = r - (r - 1) ** 2 / (n - 1), k - (k - 1) ** 2 / (n - 1)
    return float(np.sqrt(phi2c / min(kc - 1, rc - 1)))

def perm_within_61(v, strata, rng):
    out = v.copy()
    for s in np.unique(strata):
        i = np.flatnonzero(strata == s)
        out[i] = v[rng.permutation(i)]
    return out

rng_61, B_61 = np.random.RandomState(61), 1000
t0_61, rows_61 = time.time(), []
for run in RUNS:
    d = POOL[run]
    for c in range(5, 10):
        g = d[d.y == c]
        cl, cam, coil = codes_61(g.cluster), codes_61(g.cam), codes_61(g.coil)
        row = dict(run=run[11:], cls=NAMES[c], n=len(g), clusters_ge3=int((np.bincount(cl) >= 3).sum()))
        for f, v, strata, other in (("cam", cam, coil, "coil"), ("coil", coil, cam, "cam")):
            obs = vcorr_61(cl, v)
            row[f"V_{f}"] = obs
            if np.isnan(obs):
                row[f"p_{f}"] = row[f"p_{f}|{other}"] = np.nan
                continue
            plain = np.array([vcorr_61(cl, rng_61.permutation(v)) for _ in range(B_61)])
            strat = np.array([vcorr_61(cl, perm_within_61(v, strata, rng_61)) for _ in range(B_61)])
            row[f"p_{f}"] = (1 + (plain >= obs).sum()) / (B_61 + 1)
            row[f"p_{f}|{other}"] = (1 + (strat >= obs).sum()) / (B_61 + 1)
        rows_61.append(row)
df_61 = pd.DataFrame(rows_61)
print(f"{(time.time() - t0_61) / 60:.1f} min")
summ_61 = df_61.assign(ft=df_61.run.str[:2]).groupby(["cls", "ft"]).agg(
    clusters=("clusters_ge3", "median"), V_cam=("V_cam", "median"), V_coil=("V_coil", "median"),
    sig_cam=("p_cam", lambda p: int((p < 0.05).sum())), sig_cam_given_coil=("p_cam|coil", lambda p: int((p < 0.05).sum())),
    sig_coil=("p_coil", lambda p: int((p < 0.05).sum())), sig_coil_given_cam=("p_coil|cam", lambda p: int((p < 0.05).sum())))
print("median V and number of snapshots (of 5) with p < 0.05, per class and FT seed:")
print(summ_61.round(3).to_string())
g_61 = POOL[RUN][POOL[RUN].y == 5]
cpc = g_61.groupby("coil").cam.nunique()
print(f"\nsilk r0 s4: {g_61.coil.nunique()} coils, {g_61.cam.nunique()} cameras; cameras per coil: {cpc.value_counts().sort_index().to_dict()}")

0.2 min
median V and number of snapshots (of 5) with p < 0.05, per class and FT seed:
              clusters  V_cam  V_coil  sig_cam  sig_cam_given_coil  sig_coil  sig_coil_given_cam
cls       ft                                                                                    
crease    r0       2.0  0.000   0.000        0                   0         0                   0
          r1       3.0  0.122   0.526        1                   0         3                   4
          r2       3.0  0.203   0.530        1                   0         3                   3
inclusion r0       5.0  0.500   0.509        5                   5         5                   4
          r1       4.0  0.464   0.612        5                   5         5                   4
          r2       3.0  0.504   0.593        5                   5         5                   2
rolled    r0       1.0  0.591   0.378        0                   0         0                   0
          r1       1.0  0.591   0.378    

- **Kết quả**: median V và số ảnh chụp (trên 5) có p < 0.05, theo FT seed:
  - **silk** (3–4 cụm có ≥ 3 ảnh): V với camera 0.76–0.77, với cuộn 0.63–0.73. Cả bốn phép kiểm (thường, camera | cuộn, cuộn | camera) đều có ý nghĩa ở 5/5 ảnh chụp, ở cả 3 FT seed.
  - **inclusion**: V với camera 0.46–0.50, với cuộn 0.51–0.61. Camera và camera | cuộn có ý nghĩa ở 5/5; cuộn | camera ở 4, 4, 2.
  - **Đối chứng âm**:
    - rolled: 0/5 ở mọi phép kiểm (17 ảnh, 1–2 cụm);
    - crease r0: V = 0;
    - crease r1, r2: cách chia crease bám cuộn (V 0.53; cuộn có ý nghĩa ở 3/5; cuộn | camera ở 3–4/5), không bám camera (camera | cuộn 0/5);
    - waist: V cao (0.68–0.90) nhưng chỉ vài ảnh waist nằm ngoài cụm chính (median 1 cụm có ≥ 3 ảnh); camera | cuộn 0/5.
  - silk r0 s4 có 16 cuộn, 8 camera: 7 cuộn chỉ được chụp bởi 1 camera, 7 cuộn bởi 2 camera, 2 cuộn bởi 6 camera. Camera và cuộn bắt chéo đủ để tách nhau.
- **Ý nghĩa**:
  - Q09.13 cho silk và inclusion: có. k-means chia hai lớp này theo cả camera lẫn cuộn, và mỗi yếu tố vẫn có ý nghĩa sau khi giữ yếu tố kia cố định. Dự đoán "cuộn ít nhất bằng camera" đúng ở silk (cùng 5/5).
  - Đối chứng âm chỉ đạt một phần:
    - rolled và crease r0: đạt;
    - crease r1, r2: thấy liên hệ với cuộn;
    - waist: liên hệ thô với camera biến mất khi giữ cuộn cố định, nên khớp với "camera đi cùng cuộn".
  - [SUY LUẬN] Cuộn chỉ có ở pool, không có ở test (Q10.03). Cụm bám cuộn trong silk, inclusion và crease là ứng viên cụ thể cho phần điểm mất khi chuyển từ pool sang test.
  - Giới hạn: 75 tổ hợp lớp × ảnh chụp, mỗi tổ hợp 4 phép kiểm, nên phải đọc theo FT seed (n = 3), không đọc từng p.

### 6.2 Cỡ hiệu ứng trong silk và inclusion

Hai loại R² hiệu chỉnh (như `05a`):
- (i) **thành viên cụm** (one-hot) giải thích bởi camera, bởi cuộn, bởi cả hai; tách thành phần riêng camera, riêng cuộn, chung (Borcard et al. 1992; Peres-Neto et al. 2006);
- (ii) **phương sai đặc trưng thô** trong lớp giải thích bởi cách k-means chia lớp đó, bởi camera, bởi cuộn, bởi cả hai.

Số `05a` để so: trong silk, camera giải thích 0.431, cuộn 0.397, cả hai 0.607 (đặc trưng thô, r0 s4).

Dự đoán [SUY LUẬN]:
- thành viên cụm giải thích bởi cả hai khoảng 0.4–0.6, riêng camera và riêng cuộn đều dương;
- cách k-means chia silk giải thích ít phương sai đặc trưng hơn camera + cuộn, vì k-means chỉ có 2–4 nhóm cho silk.

In [10]:
# cell 9: effect sizes within silk and inclusion: (i) cluster membership (one-hot) explained by camera / coil (adjusted R^2,
#         unique and shared parts, as 05a); (ii) raw feature variance explained by the k-means split, by camera, by coil
def dummies_62(*cols):
    return np.hstack([pd.get_dummies(pd.Series(np.asarray(c)).astype(str), drop_first=True).to_numpy(float) for c in cols])

def adj_r2_62(Y, *cols):
    X = np.hstack([np.ones((len(Y), 1)), dummies_62(*cols)])
    beta, *_ = np.linalg.lstsq(X, Y, rcond=None)
    rss, tss = ((Y - X @ beta) ** 2).sum(), ((Y - Y.mean(0)) ** 2).sum()
    n, k = len(Y), X.shape[1] - 1
    return 1 - (rss / tss) * (n - 1) / (n - k - 1)

rows_62 = []
for run in RUNS:
    d, x = POOL[run], ls.pool(run)["x"].astype(np.float64)
    for c in (5, 6):
        m = (d.y == c).values
        g = d[m]
        M = pd.get_dummies(g.cluster.astype(str)).to_numpy(float)
        if M.shape[1] < 2:
            continue
        rc, rk, rb = adj_r2_62(M, g.cam), adj_r2_62(M, g.coil), adj_r2_62(M, g.cam, g.coil)
        Xc = x[m]
        rows_62.append(dict(run=run[11:], cls=NAMES[c], memb_cam_only=rb - rk, memb_coil_only=rb - rc, memb_shared=rc + rk - rb,
                            memb_both=rb, feat_kmeans_split=adj_r2_62(Xc, g.cluster), feat_cam=adj_r2_62(Xc, g.cam),
                            feat_coil=adj_r2_62(Xc, g.coil), feat_both=adj_r2_62(Xc, g.cam, g.coil)))
df_62 = pd.DataFrame(rows_62)
print(df_62.assign(ft=df_62.run.str[:2]).groupby(["cls", "ft"]).mean(numeric_only=True).round(3).to_string())

              memb_cam_only  memb_coil_only  memb_shared  memb_both  feat_kmeans_split  feat_cam  feat_coil  feat_both
cls       ft                                                                                                          
inclusion r0          0.181           0.179        0.351      0.711              0.232     0.353      0.403      0.563
          r1          0.220           0.126        0.315      0.661              0.166     0.342      0.423      0.564
          r2          0.266           0.126        0.306      0.697              0.144     0.320      0.414      0.545
silk      r0          0.274           0.116        0.484      0.874              0.344     0.431      0.397      0.607
          r1          0.235           0.196        0.493      0.924              0.391     0.423      0.398      0.615
          r2          0.254           0.192        0.458      0.905              0.367     0.420      0.382      0.599


- **Kết quả** (trung bình theo FT seed):
  - Thành viên cụm giải thích bởi camera + cuộn: silk 0.87–0.92; inclusion 0.66–0.71. Phần riêng camera 0.18–0.27, riêng cuộn 0.12–0.20, phần chung 0.31–0.49.
  - Phương sai đặc trưng trong silk: cách k-means chia silk giải thích 0.34–0.39; camera 0.42–0.43; cuộn 0.38–0.40; cả hai 0.60–0.62. Ở r0 trùng `05a` (0.431 / 0.397 / 0.607).
  - Trong inclusion: cách k-means chia 0.14–0.23; camera 0.32–0.35; cuộn 0.40–0.42; cả hai 0.55–0.56.
- **Ý nghĩa**:
  - Biết camera và cuộn của một ảnh silk thì gần như biết nó rơi vào cụm silk nào (khoảng 90% phương sai của thành viên cụm). Các cụm con của silk về thực chất là các nhóm camera/cuộn.
  - Sai dự đoán ở mức độ: dự đoán 0.4–0.6, thực tế 0.87–0.92 ở silk. Đúng ở chỗ riêng camera và riêng cuộn đều dương, và cách k-means chia lớp giải thích ít phương sai hơn camera + cuộn.

## 7. Cụm crease: r0 s2 so với r0 s4 (QD24)

### 7.1 Hai cấu hình trên cùng một pool

Hai ảnh chụp cùng FT seed r0:
- r0 s2: cụm thứ năm về crease (crease 28/32 ở cụm của nó);
- r0 s4: cụm thứ năm về inclusion.

`09-cluster` §3.5 đã cho thấy, không cần nhãn: nghiệm r0 s2 có mặt trong landscape của r0 s4 (ARI lớn nhất 0.988) nhưng hiếm (khối lượng best-of-10 chỉ 0.001); chiều ngược lại, nghiệm r0 s4 trong landscape của r0 s2 có ARI 0.998 và khối lượng 0.113.

Ở đây đặt hai cấu hình lên cùng một pool: chạy Lloyd trên pool của mỗi ảnh chụp, lần lượt từ tâm cụm của r0 s2 và của r0 s4. Ghi J, số vòng Lloyd, sở hữu cụm, best của crease và inclusion, balanced accuracy tầng 1. Kèm bảng cụm × lớp của r0 s2.

Phép kiểm: chạy từ tâm của chính ảnh chụp đó thì phải giữ nguyên nghiệm pipeline (J trùng 83113 và 82926).

Dự đoán [SUY LUẬN]: trên cả hai pool, cấu hình "inclusion" có J thấp hơn cấu hình "crease", vì nghiệm r0 s2 cách nghiệm tốt nhất của chính nó 0.52% (`09-cluster` §3.1). Tức SSE không ưa cấu hình cho crease cụm riêng, dù cấu hình đó có balanced accuracy tầng 1 cao hơn.

In [11]:
# cell 10: crease mechanism (QD24): r0 s2 (crease owns the fifth cluster) vs r0 s4 (inclusion does); each configuration is
#          run to a Lloyd fixed point on both pools from its own centers, so the two are compared on the same data
from sklearn.cluster import KMeans

def centers_71(run):
    p = ls.pool(run)
    return np.stack([p["x"][p["ids"] == k].mean(0) for k in range(p["ids"].max() + 1)])

t_71, own_71 = table_21(POOL[RUN_CREASE])
print("r0 s2, rows = clusters of the pipeline:")
print(t_71.assign(owner=own_71, size=t_71.sum(axis=1)).to_string())
rows_71 = []
for pool_run in (RUN_CREASE, RUN):
    x, d = ls.pool(pool_run)["x"], POOL[pool_run]
    for cfg_run in (RUN_CREASE, RUN):
        km = KMeans(n_clusters=5, init=centers_71(cfg_run), n_init=1).fit(x)
        sc = score.score09(d.y.values, km.labels_, NAMES, n_old=5)
        rows_71.append(dict(pool=pool_run[-5:], centers_from=cfg_run[-5:], J=round(float(km.inertia_), 1), n_iter=km.n_iter_,
                            ari_to_pipeline=round(ls.ari(km.labels_, d.cluster.values), 3),
                            owned_s_i_r_c_w=[sc[f"owned_{n}"] for n in NAMES[5:]], best_crease=round(sc["best_crease"], 2),
                            best_inclusion=round(sc["best_inclusion"], 2), bal_acc=round(sc["bal_acc_hungarian"], 3)))
print(pd.DataFrame(rows_71).to_string(index=False))

r0 s2, rows = clusters of the pipeline:
y        punching  welding  crescent  water  oil  silk  inclusion  rolled  crease  waist   owner  size
cluster                                                                                               
0               0        0         2      5    0   229         19       0       0      0    silk   255
1               0        0         0      3    1   116         47       2       4      1    silk   174
2              10       10         8      0    6     0          6      14      28     11  crease    93
3               0        0         0      0    0     0          3       1       0     82   waist    86
4               0        0         0      2    3    75         60       0       0      0    silk   140
 pool centers_from       J  n_iter  ari_to_pipeline owned_s_i_r_c_w  best_crease  best_inclusion  bal_acc
r0_s2        r0_s2 83112.7       1            1.000 [3, 0, 0, 1, 1]         0.88            0.00    0.571
r0_s2        r0_s4 82918.6 

- **Kết quả**:
  - Cụm "crease" của r0 s2 (93 ảnh, độ thuần 0.30) gồm 28 crease, 14 rolled, 11 waist, 6 inclusion và 34 ảnh cũ (punching 10, welding 10, crescent 8, oil 6). Ba cụm silk đều lẫn inclusion (19, 47, 60 ảnh). Cụm waist gần như thuần (82/86).
  - Phép kiểm đạt: chạy từ tâm của chính ảnh chụp thì Lloyd dừng ngay sau 1 vòng, ARI 1.000 với nghiệm pipeline, J trùng (83112.7 và 82926.2).
  - Hai cấu hình trên cùng một pool:

    | Pool | Cấu hình | J | Balanced accuracy tầng 1 |
    |---|---|---|---|
    | r0 s2 | crease (tâm của r0 s2) | 83112.7 | 0.571 |
    | r0 s2 | inclusion (tâm của r0 s4, 7 vòng Lloyd) | 82918.6 (thấp hơn 0.23%) | 0.326 |
    | r0 s4 | crease (tâm của r0 s2, 3 vòng Lloyd) | 83215.0 | 0.559 |
    | r0 s4 | inclusion (tâm của r0 s4) | 82926.2 (thấp hơn 0.35%) | 0.317 |

- **Ý nghĩa**:
  - Đúng như dự đoán. Trên cùng một dữ liệu, cấu hình cho crease cụm riêng có SSE cao hơn nhưng balanced accuracy tầng 1 cao gần gấp đôi. Mục tiêu SSE chọn ngược với nhãn ở đây.
  - "Cụm crease" thật ra là cụm của lớp nhỏ và ảnh cũ (crease + rolled + 34 ảnh cũ).
  - [SUY LUẬN] Đây là một ca cụ thể mà inertia thấp hơn đi cùng nhãn giả kém hơn. Nó không thay E09.07: E09.07 đo ở tầng 2, qua nhiều lần rút, với thiết kế đã chốt trước khi có số này.

## Summary (EN)

- The tier-1 table of the 15 pipeline partitions is reproduced from the snapshots: all 26 `k5_*` keys of E09.05 v2 match in 15/15. Ownership is as in QD24: waist always owns one cluster, rolled none, silk 2–3; the fifth cluster goes to crease (with rolled) or to inclusion (§1.2, §2.1).
- Misplacements in the pool are structural, not label ambiguity. Only 110/2471 (4.5%) misplaced novel images carry a box of the owner class. crease and rolled ride with waist or share one cluster with old images; inclusion merges into silk. Class sizes have CV 1.17, cluster sizes CV 0.24–0.57 (uniform effect; Xiong, Wu & Chen 2009) (§2.2).
- The k-means objective does not favour the class partition:
  - no one-step Ward swap pays: freeing a cluster costs 3645–4748, isolating rolled, crease or inclusion gains at most 1709 (§3.1);
  - Lloyd started from the five class means falls into category (c) in 15/15 snapshots (ARI to the classes 0.34–0.42), at 0.9–3.6% above the best inertia found, in a mode that best-of-10 never reaches (§4.1).
- Q09.13: within silk, camera and coil together explain 0.87–0.92 of cluster membership, and each stays significant given the other in 5/5 snapshots of every FT seed; inclusion 0.66–0.71. Controls: rolled shows nothing; crease in r1 and r2 follows coil (§6.1, §6.2). [inference] Coils differ between pool and test (Q10.03).
- On the same pool, the configuration in which inclusion owns the fifth cluster has 0.23–0.35% lower inertia than the one in which crease does, but tier-1 balanced accuracy 0.32–0.33 against 0.56–0.57 (§7.1). [inference] Lower inertia can mean worse pseudo-labels; E09.07, pre-registered before these numbers, decides at tier 2.